# Artifact: sinh ref CosyVoice bằng Kokoro

**Mục tiêu:** tạo file giọng mẫu **đúng format CosyVoice** *trước* khi chạy lab CosyVoice3.

## CosyVoice zero-shot cần gì?

| Thành phần | Yêu cầu |
|------------|---------|
| File audio | **mono**, PCM 16-bit WAV |
| Sample rate | **16000 Hz** (docs: `load_wav(..., 16000)`) |
| `prompt_text` | **Trùng** nội dung đã nói trong file ref |
| Độ dài | Vài giây, 1 speaker, không ồn |

## Notebook này làm gì?

1. Cài **Kokoro** (nhẹ — đã lab ổn)  
2. Sinh 2 ref: `bf_emma` → spk_a, `bm_george` → spk_b  
3. Resample **16 kHz**, ghi vào `voice_bank/cosy_refs/`  
4. Ghi `refs_manifest.json` + `voice_map_runtime.json` (có `prompt_text` kiểu CV3)

**Lab only** — ref tổng hợp từ Kokoro (TTS hai lần). Không sửa script IELTS. Không chốt TTS.

## Thứ tự chạy

**Cell 1 → 2 → 3 → 4 → 5**. Có thể Run all (không cần Restart kiểu Orpheus).

Sau đó mở `notebooks/colab_cosyvoice3_manifest_lab.ipynb`.


## Cell 1 — Config


In [ ]:
# CELL 1
REPO_URL = "https://github.com/phamtu2x5/ielts-script2audio.git"
BRANCH = "main"
WORKDIR = "/content/ielts-script2audio"
print("OK Cell 1")


## Cell 2 — Clone / pull repo


In [ ]:
# CELL 2
import os
from pathlib import Path

if not Path(WORKDIR).exists():
    !git clone --branch {BRANCH} {REPO_URL} {WORKDIR}
else:
    %cd {WORKDIR}
    !git pull origin {BRANCH}
%cd {WORKDIR}
assert Path("pyproject.toml").is_file()
print("OK Cell 2", os.getcwd())


## Cell 3 — Cai Kokoro + tool (Colab)


In [ ]:
# CELL 3 — light stack only (Kokoro, not CosyVoice/Orpheus)
!pip -q install -e ".[dev]" kokoro soundfile torchaudio
!apt-get -qq update && apt-get -qq install -y espeak-ng

import torch
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
from kokoro import KPipeline
print("OK Cell 3 Kokoro import")


## Cell 4 — Sinh ref 16 kHz + manifest


In [ ]:
# CELL 4 — build CosyVoice-ready refs from Kokoro
from pathlib import Path
import json
import sys

sys.path.insert(0, str(Path("scripts").resolve()))
from lab_build_cosyvoice_refs_from_kokoro import build_refs, COSY_PROMPT_SR

out_dir = Path("voice_bank/cosy_refs")
manifest = build_refs(out_dir, lang_code="b", target_sr=COSY_PROMPT_SR)

print("target_sr", COSY_PROMPT_SR)
for sp in manifest["speakers"]:
    print("---", sp["speaker_key"], sp["kokoro_voice"])
    print("wav", sp["ref_wav"])
    print("sr", sp["sample_rate_hz"], "dur", sp["duration_sec"])
    print("prompt_text:", sp["prompt_text"][:100], "...")
print("OK Cell 4")
print("files:", list(out_dir.glob("*")))


## Cell 5 — Verify format (trước khi sang CosyVoice)


In [ ]:
# CELL 5 — verify artifacts
from pathlib import Path
import json
import wave

out_dir = Path("voice_bank/cosy_refs")
man = json.loads((out_dir / "refs_manifest.json").read_text())
vmap = json.loads((out_dir / "voice_map_runtime.json").read_text())

ok = True
for sp in man["speakers"]:
    p = Path(sp["ref_wav"])
    if not p.is_file():
        print("MISSING", p)
        ok = False
        continue
    with wave.open(str(p), "rb") as wf:
        ch, sw, sr, n = wf.getnchannels(), wf.getsampwidth(), wf.getframerate(), wf.getnframes()
    print(sp["speaker_key"], "ch=", ch, "sw=", sw, "sr=", sr, "frames=", n, "sec=", round(n/sr, 2))
    if ch != 1 or sw != 2 or sr != 16000:
        print("  FORMAT FAIL — CosyVoice expects mono int16 16kHz")
        ok = False
    # prompt body must equal ref_line
    body = sp["prompt_text"].split("<|endofprompt|>")[-1]
    if body.strip() != sp["ref_line"].strip():
        print("  PROMPT mismatch body vs ref_line")
        ok = False
    # map keys
    vp = sp["voice_profile_id"]
    if vp not in vmap.get("mapping", {}):
        print("  MAP missing", vp)
        ok = False

print("voice_map keys", list(vmap.get("mapping", {})))
if ok:
    print("OK Cell 5 — artifacts san sang cho notebook CosyVoice3")
    print("Next: notebooks/colab_cosyvoice3_manifest_lab.ipynb (dung voice_bank/cosy_refs/)")
else:
    raise RuntimeError("Artifact verify failed")


## Output mong đợi

```text
voice_bank/cosy_refs/
  spk_a_ref.wav          # mono 16kHz, bf_emma
  spk_b_ref.wav          # mono 16kHz, bm_george
  refs_manifest.json     # metadata + prompt_text
  voice_map_runtime.json # vp_spk_a/b → ref + prompt_text
```

Sau đó mở `colab_cosyvoice3_manifest_lab.ipynb` (Cell ref se copy tu day).
